# Harvest and modes for one emotion (Colab, L4)

1. **Runtime > Change runtime type > L4 GPU** before running anything (changing it later restarts the VM).
2. Set `CONFIG` in the checkout cell, e.g. `configs/harvest_angry_l25.yaml`.
3. Run the cells in order. The harvest (~12 min) and modes (~25-30 min) run in the background; re-run their status cells.
4. The last cells zip `results/<run_id>` (about 20 MB) for download, and optionally copy the 2-3 GB `runs/<run_id>` to Drive.

Then, on your own machine: `scripts/judge_modes.py` (OpenRouter key) and `scripts/modes_report.py`. See the README, "Run it for another emotion".

In [ ]:
# setup (once per VM)
import torch
print("torch before install:", torch.__version__)
!pip -q install uv
!test -d /content/emotion-neurons || git clone -q https://github.com/narang99/emotion-neurons.git /content/emotion-neurons
%cd /content/emotion-neurons
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q pydantic-settings einops jaxtyping

In [ ]:
# assert the environment: a CUDA build of torch on a GPU
!python -c "import torch, transformers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'); print('transformers', transformers.__version__); assert torch.cuda.is_available() and '+cu' in torch.__version__"

In [ ]:
# checkout: the branch with the harvest code, and the emotion's config
BRANCH = "stack/harvest-happy"
CONFIG = "configs/harvest_angry_l25.yaml"

%cd /content/emotion-neurons
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q --ff-only
!uv pip install --system -q --no-deps -e .
import yaml
RUN_ID = yaml.safe_load(open(CONFIG))["run_id"]
branch = !git rev-parse --abbrev-ref HEAD
assert branch[0] == BRANCH, branch
!git log --oneline -1
print("config", CONFIG, "| run_id", RUN_ID)

In [ ]:
# harvest, in the background (~12 min on an L4)
import subprocess

def start(script, log):
    alive = subprocess.run(["pgrep", "-f", "scripts/(harvest|modes).py"], capture_output=True, text=True).stdout.split()
    if alive:
        print("a run is still alive, pids", alive)
        return
    proc = subprocess.Popen(
        f"PYTHONUNBUFFERED=1 python {script} --config {CONFIG}; echo EXIT $?",
        shell=True, cwd="/content/emotion-neurons", stdin=subprocess.DEVNULL,
        stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True,
    )
    print("started", script, "pid", proc.pid)

start("scripts/harvest.py", "/content/harvest.log")

In [ ]:
# harvest status: wait for "EXIT 0" after "[total]"
!tr '\r' '\n' < /content/harvest.log | grep -E "^\[|Error|Traceback|EXIT" | tail -12

In [ ]:
# modes, in the background, after the harvest printed EXIT 0 (~25-30 min)
assert "EXIT 0" in open("/content/harvest.log").read(), "the harvest has not finished cleanly"
start("scripts/modes.py", "/content/modes.log")

In [ ]:
# modes status: wait for "EXIT 0" after "[total]"; the clustering step prints nothing for ~10 min
!tr '\r' '\n' < /content/modes.log | grep -E "^\[|Error|Traceback|EXIT" | tail -12

In [ ]:
# download the results (about 20 MB): harvest.json, modes.json, modes/ batch files, plus both logs
assert "EXIT 0" in open("/content/modes.log").read(), "modes has not finished cleanly"
!cp /content/harvest.log /content/modes.log /content/emotion-neurons/results/{RUN_ID}/
!cd /content/emotion-neurons/results && zip -q -r /content/{RUN_ID}_results.zip {RUN_ID}
from google.colab import files
files.download(f"/content/{RUN_ID}_results.zip")

In [ ]:
# optional: keep the 2-3 GB activation pass (runs/<run_id>) on your Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/emotion-neurons && cp -r /content/emotion-neurons/runs/{RUN_ID} /content/drive/MyDrive/emotion-neurons/
!ls -la /content/drive/MyDrive/emotion-neurons/{RUN_ID}

In [ ]:
# release the VM when everything is saved (credits burn on wall-clock time)
from google.colab import runtime
runtime.unassign()